In [46]:
import xarray as xr
import numpy as np
import pandas as pd
import glob
from datetime import datetime, timedelta
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

def calculate_tercile_flags(data, var):
    # calculate terciles to cluster indices into 3 categories: -1 (negative/cold), 0 (netural), 1 (positive/warm state)
    df = data[var].to_dataframe()
    df = df.reset_index()
    df = df.set_index('time')
    tertiles = sorted(df[var].quantile([1/3, 2/3]).to_list())
    tertiles = [-float('inf')] + tertiles + [float('inf')]

    df['NAO'] = pd.cut(df[var], bins=tertiles, labels=[-1,0,1])
    return df['NAO'], tertiles

def calculate_index(data, var, tertiles=None):
    index = (data - data.mean('time'))/ np.sqrt(data.var('time'))  
    if tertiles is not None:
        nao = pd.cut(index[var].to_dataframe()[var], bins=tertiles, labels=[-1,0,1])
        index['NAO'] = xr.Dataset.from_dataframe(pd.DataFrame(nao)).zg
    else:
        nao, tertiles = calculate_tercile_flags(index, var)
        index['NAO'] = nao
        index['tertiles'] = tertiles
    index = index.rename({var: 'NAO_index'})
    return index


def calculate_nao(ds):
    ds_south = ds.sel(lat=slice(20,55), lon=slice(-90,60))
    ds_south = global_mean(ds_south.zg)
    ds_north = ds.sel(lat=slice(55,90), lon=slice(-90,60))
    ds_north = global_mean(ds_north.zg)
    nao = (ds_south - ds_north).compute()
    
    return nao.to_dataset(name='zg')

def global_mean(data, lat='lat', lon='lon'):
    weights_lat_model = np.cos(np.deg2rad(data.lat))
    try:
        mean = data.weighted(weights_lat_model).mean([lat, lon])
    except ValueError:
        mean = data.weighted(weights_lat_model).mean([lat])
    return mean

def shift_lon(ds, lon_name = 'lon'):
    # Adjust lon values to make sure they are within (-180, 180)
    ds['_longitude_adjusted'] = xr.where(
        ds[lon_name] > 180,
        ds[lon_name] - 360,
        ds[lon_name])

    # reassign the new coords to as the main lon coords
    # and sort DataArray using new coordinate values
    ds = (
        ds
        .swap_dims({lon_name: '_longitude_adjusted'})
        .sel(**{'_longitude_adjusted': sorted(ds._longitude_adjusted)})
        .drop_vars(lon_name))

    ds = ds.rename({'_longitude_adjusted': lon_name})
    return ds

In [ ]:
dt = xr.open_datatree('/sto0/data/Intermediate/LEADER/zg500_hist-volc_1850-2014_DJFonly.nc')
dt = dt.map_over_datasets(xr.DataArray.squeeze)
groups = dt.groups[1:]
dt_ens = dt.mean('member_id')

for group in groups:
    ds = dt_ens[group]
    nao = calculate_nao(ds)
    nao_index = calculate_index(nao, 'zg')
    dt_ens[group] = nao_index

    ds = dt[group]
    nao = calculate_nao(ds)
    nao_index = calculate_index(nao, 'zg', dt_ens[group]['tertiles'].values)
    dt[group] = nao_index

In [55]:
dt.sel(time='1991-12').map_over_datasets(xr.DataArray.squeeze)

<xarray.DataTree>
Group: /
├── Group: /ACCESS-ESM1-5
│       Dimensions:    (member_id: 10)
│       Coordinates:
│           plev       float64 8B 5e+04
│           source_id  <U13 52B 'ACCESS-ESM1-5'
│         * member_id  (member_id) object 80B 'r10i1p1f1' 'r1i1p1f1' ... 'r9i1p1f1'
│           time       datetime64[ns] 8B 1991-12-01
│       Data variables:
│           NAO_index  (member_id) float64 80B -0.9805 0.9004 0.386 ... 0.5248 -0.5846
│           NAO        (member_id) int64 80B -1 1 0 1 -1 0 -1 1 1 -1
├── Group: /CMCC-CM2-SR5
│       Dimensions:    (member_id: 10)
│       Coordinates:
│           plev       float64 8B 5e+04
│           source_id  <U12 48B 'CMCC-CM2-SR5'
│         * member_id  (member_id) object 80B 'r10i1p1f1' 'r11i1p1f1' ... 'r9i1p1f1'
│           time       datetime64[ns] 8B 1991-12-01
│       Data variables:
│           NAO_index  (member_id) float64 80B -0.3901 0.1625 0.4393 ... -0.1194 -0.3019
│           NAO        (member_id) int64 80B 0 0 1 0 1 1 1 1 0 0
├── Group: /CanESM5
│       Dimensions:    (member_id: 50)
│       Coordinates:
│           plev       float64 8B 5e+04
│           source_id  <U7 28B 'CanESM5'
│         * member_id  (member_id) object 400B 'r10i1p1f1' 'r10i1p2f1' ... 'r9i1p2f1'
│           time       datetime64[ns] 8B 1991-12-01
│       Data variables:
│           NAO_index  (member_id) float64 400B -1.248 0.7428 1.104 ... 1.389 -0.07112
│           NAO        (member_id) int64 400B -1 1 1 -1 0 0 1 0 -1 ... 1 1 1 1 -1 0 1 0
...
├── Group: /MPI-ESM1-2-LR
│       Dimensions:    (member_id: 30)
│       Coordinates:
│           plev       float64 8B 5e+04
│           source_id  <U13 52B 'MPI-ESM1-2-LR'
│         * member_id  (member_id) object 240B 'r10i1p1f2' 'r11i1p1f2' ... 'r9i1p1f2'
│           time       datetime64[ns] 8B 1991-12-01
│       Data variables:
│           NAO_index  (member_id) float64 240B -0.08776 -2.026 ... -0.01561 -0.8874
│           NAO        (member_id) int64 240B 0 -1 1 1 1 -1 1 -1 ... 1 -1 0 1 0 -1 0 -1
├── Group: /MRI-ESM2-0
│       Dimensions:    (member_id: 5)
│       Coordinates:
│           plev       float64 8B 5e+04
│           source_id  <U10 40B 'MRI-ESM2-0'
│         * member_id  (member_id) object 40B 'r1i1p1f1' 'r2i1p1f1' ... 'r5i1p1f1'
│           time       datetime64[ns] 8B 1991-12-01
│       Data variables:
│           NAO_index  (member_id) float64 40B 0.8936 0.6534 0.6656 0.3917 1.451
│           NAO        (member_id) int64 40B 1 1 1 0 1
└── Group: /NorESM2-LM
        Dimensions:    (member_id: 20)
        Coordinates:
            plev       float64 8B 5e+04
            source_id  <U10 40B 'NorESM2-LM'
          * member_id  (member_id) object 160B 'r10i1p1f1' 'r11i1p1f1' ... 'r9i1p1f1'
            time       datetime64[ns] 8B 1991-12-01
        Data variables:
            NAO_index  (member_id) float64 160B 0.4141 -0.5235 ... -0.3546 0.01916
            NAO        (member_id) int64 160B 0 -1 -1 0 -1 -1 1 1 ... 0 0 0 -1 -1 0 0 0

In [ ]:
''